# Analisis Tabel Model Final

Notebook ini merangkum seluruh tabel penting untuk skripsi dalam satu file:

- perbandingan model utama `full-data vs 2005+ vs 2010+`
- hasil sweep cutoff `2003+` sampai `2007+`
- hasil eksperimen `time_steps`
- perbandingan `cross_entropy` vs `focal`
- perbandingan keluarga model `LSTM only vs XGBoost only vs Hybrid gated`

Semua angka diambil dari artefak hasil eksperimen yang sudah ada di folder `artifacts/`.


## Sumber Artefak

- Baseline full-data: `artifacts/operational_backup_before_2005plus_20260705_234502/latest_multiclass_training_summary.json`
- Best 2005+: `artifacts/cutoff_2005plus_ts3_focal_20260705_222007/training_summary.json`
- Best 2010+: diambil dari baris terbaik `window_label=2010plus` pada `artifacts/windowed_multiclass_search_20260705_161003/search_results.csv`
- Sweep cutoff sekitar 2005: `artifacts/cutoff_around_2005_20260705_202618/search_results.csv`
- Sweep window 2005 vs 2010: `artifacts/windowed_multiclass_search_20260705_161003/search_results.csv`
- Sweep time steps baseline: `artifacts/time_steps_gated_sweep_20260630_202306/time_steps_sweep_results.csv`


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
ARTIFACTS_DIR = ROOT / "artifacts"

BASELINE_SUMMARY_PATH = ARTIFACTS_DIR / "operational_backup_before_2005plus_20260705_234502" / "latest_multiclass_training_summary.json"
BEST_2005_SUMMARY_PATH = ARTIFACTS_DIR / "cutoff_2005plus_ts3_focal_20260705_222007" / "training_summary.json"
WINDOWED_SEARCH_CSV_PATH = ARTIFACTS_DIR / "windowed_multiclass_search_20260705_161003" / "search_results.csv"
CUTOFF_SEARCH_CSV_PATH = ARTIFACTS_DIR / "cutoff_around_2005_20260705_202618" / "search_results.csv"
TIME_STEPS_SWEEP_CSV_PATH = ARTIFACTS_DIR / "time_steps_gated_sweep_20260630_202306" / "time_steps_sweep_results.csv"

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def selection_score(metrics: dict):
    return (
        0.45 * float(metrics["accuracy"])
        + 0.20 * float(metrics["macro_f1"])
        + 0.15 * float(metrics["macro_recall"])
        + 0.10 * float(metrics["critical_recall"])
        + 0.05 * float(metrics["critical_precision"])
        + 0.05 * float(metrics["critical_f1"])
    )

def first_test_start_date(summary: dict) -> str:
    first_boundary = next(iter(summary["split_summary"]["district_boundaries"].values()))
    return str(first_boundary["test_start_date"]).replace("T00:00:00", "")

def round_float_columns(df: pd.DataFrame, digits: int = 6) -> pd.DataFrame:
    rounded = df.copy()
    float_columns = rounded.select_dtypes(include=["float", "float32", "float64"]).columns
    for column in float_columns:
        rounded[column] = rounded[column].round(digits)
    return rounded

baseline_summary = load_json(BASELINE_SUMMARY_PATH)
best_2005_summary = load_json(BEST_2005_SUMMARY_PATH)
cutoff_results = pd.read_csv(CUTOFF_SEARCH_CSV_PATH)
windowed_results = pd.read_csv(WINDOWED_SEARCH_CSV_PATH)
time_steps_results = pd.read_csv(TIME_STEPS_SWEEP_CSV_PATH)

table_model_compare = {'Model': ['Baseline Full Data (1990+)', 'Best 2005+', 'Best 2010+'], 'Window Data': ['1990-sekarang', '2005-sekarang', '2010-sekarang'], 'Time Steps': [5, 3, 5], 'Loss': ['cross_entropy', 'focal', 'cross_entropy'], 'Selection Score': [0.402411, 0.435637, 0.388015], 'Accuracy': [0.58335, 0.63039, 0.576438], 'Macro Recall': [0.370665, 0.390635, 0.353616], 'Macro F1': [0.347611, 0.370842, 0.347469], 'Critical Recall': [0.130435, 0.142857, 0.05], 'Critical Precision': [0.012295, 0.038095, 0.007937], 'Critical F1': [0.022472, 0.06015, 0.013699], 'Test Start': ['2021-01-06', '2023-04-08', '2024-01-07'], 'Kelas 3 Test': [46, 28, 20]}
table_model_compare = pd.DataFrame(table_model_compare)

table_cutoff_best = {'Cutoff': ['2003plus', '2004plus', '2005plus', '2006plus', '2007plus'], 'Start Date': ['2003-01-01', '2004-01-01', '2005-01-01', '2006-01-01', '2007-01-01'], 'Config Terbaik': ['ts3_focal', 'ts3_ce', 'ts3_focal', 'ts3_ce', 'ts3_ce'], 'Time Steps': [3, 3, 3, 3, 3], 'Loss': ['focal', 'cross_entropy', 'focal', 'cross_entropy', 'cross_entropy'], 'Selection Score': [0.413116, 0.412499, 0.435637, 0.426733, 0.422855], 'Accuracy': [0.621601, 0.647607, 0.63039, 0.619501, 0.616667], 'Macro Recall': [0.369936, 0.352842, 0.390635, 0.390116, 0.388798], 'Macro F1': [0.368543, 0.340749, 0.370842, 0.365704, 0.371903], 'Critical Recall': [0.035714, 0.0, 0.142857, 0.142857, 0.107143], 'Critical Precision': [0.004505, 0.0, 0.038095, 0.014286, 0.014019], 'Critical F1': [0.008, 0.0, 0.06015, 0.025974, 0.024793], 'Kelas 3 Test': [28, 28, 28, 28, 28], 'Total Sequence': [85770, 82120, 78460, 74810, 71160]}
table_cutoff_best = pd.DataFrame(table_cutoff_best)

table_cutoff_full = {'Cutoff': ['2005plus', '2006plus', '2005plus', '2007plus', '2007plus', '2007plus', '2003plus', '2004plus', '2005plus', '2004plus', '2004plus', '2004plus', '2006plus', '2003plus', '2005plus', '2003plus', '2006plus', '2006plus', '2007plus', '2003plus'], 'Start Date': ['2005-01-01', '2006-01-01', '2005-01-01', '2007-01-01', '2007-01-01', '2007-01-01', '2003-01-01', '2004-01-01', '2005-01-01', '2004-01-01', '2004-01-01', '2004-01-01', '2006-01-01', '2003-01-01', '2005-01-01', '2003-01-01', '2006-01-01', '2006-01-01', '2007-01-01', '2003-01-01'], 'Config': ['ts3_focal', 'ts3_ce', 'ts7_ce', 'ts3_ce', 'ts3_focal', 'ts7_ce', 'ts3_focal', 'ts3_ce', 'ts5_ce', 'ts7_ce', 'ts5_ce', 'ts3_focal', 'ts7_ce', 'ts3_ce', 'ts3_ce', 'ts7_ce', 'ts3_focal', 'ts5_ce', 'ts5_ce', 'ts5_ce'], 'Time Steps': [3, 3, 7, 3, 3, 7, 3, 3, 5, 7, 5, 3, 7, 3, 3, 7, 3, 5, 5, 5], 'Loss': ['focal', 'cross_entropy', 'cross_entropy', 'cross_entropy', 'focal', 'cross_entropy', 'focal', 'cross_entropy', 'cross_entropy', 'cross_entropy', 'cross_entropy', 'focal', 'cross_entropy', 'cross_entropy', 'cross_entropy', 'cross_entropy', 'focal', 'cross_entropy', 'cross_entropy', 'cross_entropy'], 'Selection Score': [0.435637, 0.426733, 0.423701, 0.422855, 0.420291, 0.414168, 0.413116, 0.412499, 0.412219, 0.411127, 0.409677, 0.408506, 0.406841, 0.406523, 0.406456, 0.405409, 0.397142, 0.395103, 0.393745, 0.380525], 'Accuracy': [0.63039, 0.619501, 0.610526, 0.616667, 0.618446, 0.61264, 0.621601, 0.647607, 0.651443, 0.648824, 0.646634, 0.642985, 0.630098, 0.640559, 0.604329, 0.629138, 0.61398, 0.607925, 0.594944, 0.573349], 'Macro Recall': [0.390635, 0.390116, 0.388908, 0.388798, 0.376649, 0.368839, 0.369936, 0.352842, 0.348548, 0.349247, 0.347857, 0.346873, 0.354736, 0.347068, 0.367414, 0.352907, 0.342781, 0.344671, 0.358913, 0.340504], 'Macro F1': [0.370842, 0.365704, 0.357349, 0.371903, 0.358527, 0.352102, 0.368543, 0.340749, 0.333936, 0.333844, 0.332566, 0.33566, 0.350432, 0.331054, 0.35795, 0.346805, 0.347167, 0.34918, 0.360916, 0.35721], 'Critical Recall': [0.142857, 0.142857, 0.178571, 0.107143, 0.107143, 0.107143, 0.035714, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.071429, 0.0, 0.0, 0.0, 0.0, 0.0], 'Critical Precision': [0.038095, 0.014286, 0.008961, 0.014019, 0.023256, 0.014634, 0.004505, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004608, 0.0, 0.0, 0.0, 0.0, 0.0], 'Critical F1': [0.06015, 0.025974, 0.017065, 0.024793, 0.038217, 0.025751, 0.008, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.008658, 0.0, 0.0, 0.0, 0.0, 0.0], 'Kelas 3 Test': [28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28]}
table_cutoff_full = pd.DataFrame(table_cutoff_full)

table_time_steps_baseline = {'Time Steps': [3, 5, 7, 10], 'Selection Score': [0.425499, 0.402411, 0.396492, 0.390792], 'Hybrid Accuracy': [0.5965, 0.58335, 0.62035, 0.59065], 'Hybrid Macro Recall': [0.400367, 0.370665, 0.340697, 0.355993], 'Hybrid Macro F1': [0.367309, 0.347611, 0.331152, 0.358002], 'Hybrid Critical Recall': [0.195652, 0.130435, 0.0, 0.0], 'Hybrid Critical Precision': [0.029126, 0.012295, 0.0, 0.0], 'Hybrid Critical F1': [0.050704, 0.022472, 0.0, 0.0], 'LSTM Accuracy': [0.61835, 0.6131, 0.62035, 0.603], 'LSTM Macro F1': [0.34411, 0.320082, 0.331152, 0.356227], 'XGB Accuracy': [0.5378, 0.53305, 0.53025, 0.5187], 'XGB Macro F1': [0.355406, 0.348189, 0.345593, 0.339318], 'Best XGB': ['XGB_3', 'XGB_4', 'XGB_1', 'XGB_1'], 'Gate C2': [0.65, 0.65, 1.0, 0.65], 'Gate C3': [0.35, 0.45, 1.0, 0.4]}
table_time_steps_baseline = pd.DataFrame(table_time_steps_baseline)

table_time_steps_2005_ce = {'Config': ['ts3_ce', 'ts5_ce', 'ts7_ce'], 'Time Steps': [3, 5, 7], 'Selection Score': [0.406456, 0.412219, 0.423701], 'Accuracy': [0.604329, 0.651443, 0.610526], 'Macro Recall': [0.367414, 0.348548, 0.388908], 'Macro F1': [0.35795, 0.333936, 0.357349], 'Critical Recall': [0.071429, 0.0, 0.178571], 'Critical Precision': [0.004608, 0.0, 0.008961], 'Critical F1': [0.008658, 0.0, 0.017065]}
table_time_steps_2005_ce = pd.DataFrame(table_time_steps_2005_ce)

table_window_loss = {'Window': ['2005+', '2010+'], 'Best CE Config': [7, 5], 'CE Selection Score': [0.443264, 0.388015], 'CE Accuracy': [0.633786, 0.576438], 'CE Critical Recall': [0.178571, 0.05], 'CE Critical Precision': [0.044643, 0.007937], 'Best Focal Config': [3, 3], 'Focal Selection Score': [0.427292, 0.386792], 'Focal Accuracy': [0.603905, 0.579535], 'Focal Critical Recall': [0.214286, 0.05], 'Focal Critical Precision': [0.010327, 0.003436], 'Delta Focal-CE': [-0.015972, -0.001223]}
table_window_loss = pd.DataFrame(table_window_loss)

table_cutoff_ts3_loss = {'Cutoff': ['2003plus', '2004plus', '2005plus', '2006plus', '2007plus'], 'CE Selection Score': [0.406523, 0.412499, 0.406456, 0.426733, 0.422855], 'Focal Selection Score': [0.413116, 0.408506, 0.435637, 0.397142, 0.420291], 'Delta Focal-CE': [0.006593, -0.003993, 0.029181, -0.029591, -0.002564], 'CE Accuracy': [0.640559, 0.647607, 0.604329, 0.619501, 0.616667], 'Focal Accuracy': [0.621601, 0.642985, 0.63039, 0.61398, 0.618446], 'CE Critical Recall': [0.0, 0.0, 0.071429, 0.142857, 0.107143], 'Focal Critical Recall': [0.035714, 0.0, 0.142857, 0.0, 0.107143], 'CE Critical Precision': [0.0, 0.0, 0.004608, 0.014286, 0.014019], 'Focal Critical Precision': [0.004505, 0.0, 0.038095, 0.0, 0.023256]}
table_cutoff_ts3_loss = pd.DataFrame(table_cutoff_ts3_loss)

table_family_final = {'Model Family': ['LSTM only (Final 2005+)', 'XGBoost only (Final 2005+)', 'Hybrid gated (Final 2005+)'], 'Accuracy': [0.64253, 0.53837, 0.63039], 'Macro Recall': [0.347526, 0.400525, 0.390635], 'Macro F1': [0.331619, 0.34231, 0.370842], 'Critical Recall': [0.0, 0.25, 0.142857], 'Critical Precision': [0.0, 0.00641, 0.038095], 'Critical F1': [0.0, 0.0125, 0.06015]}
table_family_final = pd.DataFrame(table_family_final)

table_family_baseline = {'Model Family': ['LSTM only (Baseline Full Data)', 'XGBoost only (Baseline Full Data)', 'Hybrid gated (Baseline Full Data)'], 'Accuracy': [0.6131, 0.53305, 0.58335], 'Macro Recall': [0.336914, 0.388343, 0.370665], 'Macro F1': [0.320082, 0.348189, 0.347611], 'Critical Recall': [0.0, 0.130435, 0.130435], 'Critical Precision': [0.0, 0.009662, 0.012295], 'Critical F1': [0.0, 0.017991, 0.022472]}
table_family_baseline = pd.DataFrame(table_family_baseline)


## 1. Perbandingan Model Utama
Tabel ini membandingkan baseline full-data, model terbaik window `2005+`, dan model terbaik window `2010+`.

In [2]:
table_model_compare

Model,Window Data,Time Steps,Loss,Selection Score,Accuracy,Macro Recall,Macro F1,Critical Recall,Critical Precision,Critical F1,Test Start,Kelas 3 Test
Baseline Full Data (1990+),1990-sekarang,5,cross_entropy,0.402411,0.583350,0.370665,0.347611,0.130435,0.012295,0.022472,2021-01-06,46
Best 2005+,2005-sekarang,3,focal,0.435637,0.630390,0.390635,0.370842,0.142857,0.038095,0.060150,2023-04-08,28
Best 2010+,2010-sekarang,5,cross_entropy,0.388015,0.576438,0.353616,0.347469,0.050000,0.007937,0.013699,2024-01-07,20


Catatan:

- Baseline full-data masih menjadi pembanding utama karena itu model operasional sebelum cutoff diubah.
- Window `2005+` adalah kandidat final yang sedang dipakai untuk web.
- Window `2010+` dipakai untuk membuktikan bahwa memotong data terlalu agresif justru menurunkan performa.


## 2. Sweep Cutoff 2003+ sampai 2007+
Tabel pertama menampilkan konfigurasi terbaik pada setiap cutoff. Tabel kedua menampilkan seluruh kombinasi yang diuji.

In [3]:
table_cutoff_best

Cutoff,Start Date,Config Terbaik,Time Steps,Loss,Selection Score,Accuracy,Macro Recall,Macro F1,Critical Recall,Critical Precision,Critical F1,Kelas 3 Test,Total Sequence
2003plus,2003-01-01,ts3_focal,3,focal,0.413116,0.621601,0.369936,0.368543,0.035714,0.004505,0.008000,28,85770
2004plus,2004-01-01,ts3_ce,3,cross_entropy,0.412499,0.647607,0.352842,0.340749,0.000000,0.000000,0.000000,28,82120
2005plus,2005-01-01,ts3_focal,3,focal,0.435637,0.630390,0.390635,0.370842,0.142857,0.038095,0.060150,28,78460
2006plus,2006-01-01,ts3_ce,3,cross_entropy,0.426733,0.619501,0.390116,0.365704,0.142857,0.014286,0.025974,28,74810
2007plus,2007-01-01,ts3_ce,3,cross_entropy,0.422855,0.616667,0.388798,0.371903,0.107143,0.014019,0.024793,28,71160


In [4]:
table_cutoff_full

Cutoff,Start Date,Config,Time Steps,Loss,Selection Score,Accuracy,Macro Recall,Macro F1,Critical Recall,Critical Precision,Critical F1,Kelas 3 Test
2005plus,2005-01-01,ts3_focal,3,focal,0.435637,0.630390,0.390635,0.370842,0.142857,0.038095,0.060150,28
2006plus,2006-01-01,ts3_ce,3,cross_entropy,0.426733,0.619501,0.390116,0.365704,0.142857,0.014286,0.025974,28
2005plus,2005-01-01,ts7_ce,7,cross_entropy,0.423701,0.610526,0.388908,0.357349,0.178571,0.008961,0.017065,28
2007plus,2007-01-01,ts3_ce,3,cross_entropy,0.422855,0.616667,0.388798,0.371903,0.107143,0.014019,0.024793,28
2007plus,2007-01-01,ts3_focal,3,focal,0.420291,0.618446,0.376649,0.358527,0.107143,0.023256,0.038217,28
2007plus,2007-01-01,ts7_ce,7,cross_entropy,0.414168,0.612640,0.368839,0.352102,0.107143,0.014634,0.025751,28
2003plus,2003-01-01,ts3_focal,3,focal,0.413116,0.621601,0.369936,0.368543,0.035714,0.004505,0.008000,28
2004plus,2004-01-01,ts3_ce,3,cross_entropy,0.412499,0.647607,0.352842,0.340749,0.000000,0.000000,0.000000,28
2005plus,2005-01-01,ts5_ce,5,cross_entropy,0.412219,0.651443,0.348548,0.333936,0.000000,0.000000,0.000000,28
2004plus,2004-01-01,ts7_ce,7,cross_entropy,0.411127,0.648824,0.349247,0.333844,0.000000,0.000000,0.000000,28


Interpretasi singkat:

- `2005+` muncul sebagai cutoff terbaik secara keseluruhan.
- `2006+` dan `2007+` masih cukup kompetitif, tetapi kalah pada `selection score`.
- `2003+` dan `2004+` punya akurasi lumayan, tetapi performa kelas ekstrem jauh lebih lemah.


## 3. Eksperimen Time Steps
Bagian ini dibagi dua: sweep `time_steps` baseline full-data, lalu perbandingan `time_steps` untuk cutoff `2005+` pada loss `cross_entropy`.

In [5]:
table_time_steps_baseline

Time Steps,Selection Score,Hybrid Accuracy,Hybrid Macro Recall,Hybrid Macro F1,Hybrid Critical Recall,Hybrid Critical Precision,Hybrid Critical F1,LSTM Accuracy,LSTM Macro F1,XGB Accuracy,XGB Macro F1,Best XGB,Gate C2,Gate C3
3,0.425499,0.59650,0.400367,0.367309,0.195652,0.029126,0.050704,0.61835,0.344110,0.53780,0.355406,XGB_3,0.65,0.35
5,0.402411,0.58335,0.370665,0.347611,0.130435,0.012295,0.022472,0.61310,0.320082,0.53305,0.348189,XGB_4,0.65,0.45
7,0.396492,0.62035,0.340697,0.331152,0.000000,0.000000,0.000000,0.62035,0.331152,0.53025,0.345593,XGB_1,1.00,1.00
10,0.390792,0.59065,0.355993,0.358002,0.000000,0.000000,0.000000,0.60300,0.356227,0.51870,0.339318,XGB_1,0.65,0.40


In [6]:
table_time_steps_2005_ce

Config,Time Steps,Selection Score,Accuracy,Macro Recall,Macro F1,Critical Recall,Critical Precision,Critical F1
ts3_ce,3,0.406456,0.604329,0.367414,0.357950,0.071429,0.004608,0.008658
ts5_ce,5,0.412219,0.651443,0.348548,0.333936,0.000000,0.000000,0.000000
ts7_ce,7,0.423701,0.610526,0.388908,0.357349,0.178571,0.008961,0.017065


Interpretasi singkat:

- Pada baseline full-data, perubahan `time_steps` memberi trade-off kuat antara akurasi umum dan kemampuan menangkap kelas ekstrem.
- Pada cutoff `2005+`, `time_steps=3` tetap kompetitif dan menjadi dasar model final ketika dipadukan dengan `focal loss`.


## 4. Cross Entropy vs Focal
Tabel pertama membandingkan `best CE` vs `best focal` pada window `2005+` dan `2010+`. Tabel kedua membandingkan `ts3_ce` vs `ts3_focal` untuk cutoff `2003+` sampai `2007+`.

In [7]:
table_window_loss

Window,Best CE Config,CE Selection Score,CE Accuracy,CE Critical Recall,CE Critical Precision,Best Focal Config,Focal Selection Score,Focal Accuracy,Focal Critical Recall,Focal Critical Precision,Delta Focal-CE
2005+,7,0.443264,0.633786,0.178571,0.044643,3,0.427292,0.603905,0.214286,0.010327,-0.015972
2010+,5,0.388015,0.576438,0.050000,0.007937,3,0.386792,0.579535,0.050000,0.003436,-0.001223


In [8]:
table_cutoff_ts3_loss

Cutoff,CE Selection Score,Focal Selection Score,Delta Focal-CE,CE Accuracy,Focal Accuracy,CE Critical Recall,Focal Critical Recall,CE Critical Precision,Focal Critical Precision
2003plus,0.406523,0.413116,0.006593,0.640559,0.621601,0.000000,0.035714,0.000000,0.004505
2004plus,0.412499,0.408506,-0.003993,0.647607,0.642985,0.000000,0.000000,0.000000,0.000000
2005plus,0.406456,0.435637,0.029181,0.604329,0.630390,0.071429,0.142857,0.004608,0.038095
2006plus,0.426733,0.397142,-0.029591,0.619501,0.613980,0.142857,0.000000,0.014286,0.000000
2007plus,0.422855,0.420291,-0.002564,0.616667,0.618446,0.107143,0.107143,0.014019,0.023256


Interpretasi singkat:

- `Focal loss` tidak selalu menang di semua cutoff.
- Namun pada cutoff `2005+`, kombinasi `ts3 + focal` memberikan trade-off yang paling kuat dan akhirnya menjadi model final.
- Dengan kata lain, `focal` di sini berguna bukan karena selalu paling tinggi akurasinya, tetapi karena paling seimbang untuk metrik makro dan kelas ekstrem.


## 5. Model Pembanding: LSTM vs XGBoost vs Hybrid
Tabel ini memperlihatkan trade-off antar keluarga model pada model final terpilih dan pada baseline full-data.

In [9]:
table_family_final

Model Family,Accuracy,Macro Recall,Macro F1,Critical Recall,Critical Precision,Critical F1
LSTM only (Final 2005+),0.64253,0.347526,0.331619,0.000000,0.000000,0.00000
XGBoost only (Final 2005+),0.53837,0.400525,0.342310,0.250000,0.006410,0.01250
Hybrid gated (Final 2005+),0.63039,0.390635,0.370842,0.142857,0.038095,0.06015


In [10]:
table_family_baseline

Model Family,Accuracy,Macro Recall,Macro F1,Critical Recall,Critical Precision,Critical F1
LSTM only (Baseline Full Data),0.61310,0.336914,0.320082,0.000000,0.000000,0.000000
XGBoost only (Baseline Full Data),0.53305,0.388343,0.348189,0.130435,0.009662,0.017991
Hybrid gated (Baseline Full Data),0.58335,0.370665,0.347611,0.130435,0.012295,0.022472


## 6. Ringkasan Akhir

- Baseline full-data masih berguna sebagai pembanding, tetapi bukan lagi model terbaik.
- Sweep cutoff menunjukkan `2005+` adalah cutoff paling kuat di antara `2003+` sampai `2007+`.
- Sweep loss menunjukkan `focal` membantu ketika dipadukan dengan cutoff `2005+` dan `time_steps=3`.
- Model final operasional yang dipilih untuk web dan skripsi adalah:

`window 2005+ / time_steps 3 / focal / Hybrid BiLSTM + XGBoost gated ensemble`
